# Coffee shop sales: all variables, constant set to 0

**Multiple Regression** · a worked example from the StatNex teaching guide

**The story.** A coffee chain compares 20 of its shops. Some shops sit on busy streets, some have more seats, and some have big social-media followings. Head office wants to know which of these actually drives monthly sales.

**The question.** Once you know a shop's foot traffic and seating, does its social-media following still tell you anything about its sales?

**To run it**, choose **Runtime → Run all**. The first cells save the table in this session. The rest is the calculation StatNex runs, written out in Python. It does not use StatNex, and every number it prints was checked against the app's.

## The table

This cell saves the table `09_multivariate/multiple_regression.csv` (20 rows) as `data.csv`, the file the code reads.

In [ ]:
%%writefile data.csv
No.,Monthly sales ($000),Foot traffic (thousands/month),Seats,Social followers (thousands)
1.0,42.3,6.0,39.0,4.7
2.0,39.3,8.7,11.0,6.5
3.0,21.2,5.0,11.0,0.3
4.0,54.2,11.6,46.0,3.7
5.0,36.4,10.5,14.0,3.7
6.0,54.5,15.5,49.0,12.8
7.0,44.3,10.7,44.0,6.2
8.0,37.2,5.0,45.0,5.2
9.0,37.7,6.5,27.0,6.9
10.0,37.4,6.8,40.0,1.6
11.0,47.8,10.7,42.0,6.6
12.0,37.4,9.9,28.0,3.1
13.0,62.3,15.8,48.0,8.4
14.0,57.9,13.5,51.0,3.0
15.0,61.1,12.5,51.0,11.4
16.0,44.8,8.1,49.0,1.5
17.0,52.9,14.0,10.0,6.1
18.0,65.6,14.7,58.0,8.7
19.0,38.4,7.9,19.0,2.7
20.0,39.0,9.3,16.0,4.0

## Libraries

In [ ]:
import numpy as np
import pandas as pd
import statsmodels.api as sm
from scipy import stats
from statsmodels.stats.diagnostic import het_breuschpagan, het_white
from statsmodels.stats.outliers_influence import variance_inflation_factor
from statsmodels.stats.stattools import durbin_watson

pd.set_option("display.width", 160)
pd.set_option("display.max_columns", 20)

## The data

In [ ]:
# Text in a numeric column counts as missing, as it does in the app.
data = pd.read_csv("data.csv", keep_default_na=False, na_values=[""])
response = "Monthly sales ($000)"
predictors = ["Foot traffic (thousands/month)", "Seats", "Social followers (thousands)"]
reference = []
columns = [*predictors, *reference, response]
frame = data[columns].apply(pd.to_numeric, errors="coerce")
confidence = 95 / 100

## Cases

In [ ]:
# A case is used only when the response and every explanatory variable are present.
y_ok = frame[response].notna()
x_ok = frame[predictors].notna().all(axis=1)
valid = frame[y_ok & x_ok]
n = len(valid)
summary = pd.DataFrame(
    {"n": [(y_ok & x_ok).sum(), (~y_ok & x_ok).sum(), (y_ok & ~x_ok).sum(),
           (~y_ok & ~x_ok).sum(), len(frame)]},
    index=["valid", "response missing", "explanatory missing", "both missing", "total"],
)
summary["%"] = summary["n"] / len(frame)
print(summary)

## Descriptive statistics and correlation matrix

In [ ]:
# The correlation matrix shows each pair's straight-line link on its own, before the
# regression holds the other variables fixed.
descriptive = pd.DataFrame({
    "n": n, "mean": valid[columns].mean(), "variance": valid[columns].var(ddof=1),
    "SD": valid[columns].std(ddof=1), "min": valid[columns].min(), "max": valid[columns].max(),
})
correlation = valid[columns].corr()
print()
print(descriptive)
print()
print(correlation)


# Least squares of the response on `names` with the constant fixed at 0, so the line
# goes through the origin. statsmodels then takes R², the F test and the sums of
# squares about 0 rather than about the mean (uncentred), as the app does: they cannot
# be compared with those of a model that has a constant.
def fit(names):
    return sm.OLS(valid[response].to_numpy(float), valid[names].to_numpy(float)).fit()


# R, adjusted R, R², adjusted R², the Durbin-Watson ratio and AIC. R² is measured
# against predicting 0 for every case, not the mean. Durbin-Watson compares each
# residual with the next one (near 2: no pattern from row to row); it means something
# only when the rows have an order, such as time. AIC: lower is better, for models of
# the same response on the same cases. Here it is n·ln(SSe/n) + 2k, the textbook
# regression form with k coefficients (there is no constant), as R's extractAIC()
# gives it. statsmodels' model.aic uses the full normal log-likelihood and comes out
# larger by n·(ln 2π + 1) for every model, so the two rank models alike.
def accuracy(model, k):
    return [np.sqrt(model.rsquared), np.sqrt(max(model.rsquared_adj, 0.0)), model.rsquared,
            model.rsquared_adj, durbin_watson(model.resid), n * np.log(model.ssr / n) + 2 * k]


# Regression, error and total sums of squares with their df, MS, F and P, about 0: the
# total is the uncentred Σy² on n df, and the error has n - k df.
def anova_rows(model, k):
    ss_total, ss_error = model.uncentered_tss, model.ssr
    return [[ss_total - ss_error, k, (ss_total - ss_error) / k, model.fvalue, model.f_pvalue],
            [ss_error, n - k, ss_error / (n - k), np.nan, np.nan],
            [ss_total, n, np.nan, np.nan, np.nan]]


# One row per variable; there is no constant. For each: the coefficient b (the change
# in the response per unit of the variable, the others held fixed), its standard
# error, the standardised coefficient b·SD(x)/SD(y) (comparable across variables), the
# confidence interval from conf_int(), F = t², t = b/SE and its P value (H0: b = 0),
# the simple correlation with the response, the partial correlation from t and the
# error df, n - k, and the tolerance and VIF. The VIF (variance inflation factor) says
# how much the coefficient's variance is inflated because its variable overlaps with
# the others: 1 is none, and above 10 is the usual rule-of-thumb warning; tolerance
# is 1/VIF. variance_inflation_factor() regresses the variable on the others; that
# auxiliary regression keeps a constant of its own, as the app's does, which gives
# the centred VIF.
def coefficient_table(model, names):
    k = len(names)
    interval = model.conf_int(alpha=1 - confidence)
    exog = sm.add_constant(model.model.exog, has_constant="add")
    rows = []
    for j, name in enumerate(names):
        b, se, t = model.params[j], model.bse[j], model.tvalues[j]
        vif = variance_inflation_factor(exog, j + 1)
        rows.append([b, se, b * valid[name].std() / valid[response].std(), *interval[j],
                     t**2, t, model.pvalues[j],
                     valid[response].corr(valid[name]), t / np.sqrt(t**2 + n - k),
                     1 / vif, vif])
    return pd.DataFrame(rows, index=names,
                        columns=["b", "SE", "beta", "lower", "upper", "F", "t", "P",
                                 "simple r", "partial r", "tolerance", "VIF"])

## All variables

In [ ]:
# Every explanatory variable goes into the model; there is no selection.
selected = list(predictors)

## The model

In [ ]:
# model.summary() would print statsmodels' own report of the same fit.
model = fit(selected)
k = len(selected)
fit_accuracy = pd.DataFrame([accuracy(model, k)],
                            columns=["R", "adjusted R", "R2", "adjusted R2", "DW", "AIC"])
anova = pd.DataFrame(anova_rows(model, k), index=["regression", "error", "total"],
                     columns=["SS", "df", "MS", "F", "P"])
coefficients = coefficient_table(model, selected)
print()
print(fit_accuracy.to_string(index=False))
print()
print(anova)
print()
print(coefficients)

## Predicted values, residuals and diagnostics

In [ ]:
# For each case: the predicted value, the residual (observed - predicted), the
# standardised residual (the residual over the residual SD), the studentised residual,
# which also allows for the case's leverage (beyond about +-2 is worth a look), Cook's
# distance (how much the whole fit moves if the case is dropped) and the leverage
# (how unusual the case's explanatory values are).
influence = model.get_influence()
predictions = pd.DataFrame({
    "no": np.arange(1, n + 1), "observed": valid[response].to_numpy(float),
    "predicted": model.fittedvalues, "residual": model.resid,
    "standardised": model.resid / np.sqrt(model.mse_resid),
    "studentised": influence.resid_studentized_internal,
    "Cook": influence.cooks_distance[0], "leverage": influence.hat_matrix_diag,
})
print()
print(predictions.to_string(index=False))

## Interval estimates of the predicted values

In [ ]:
# At each case's values: the confidence interval for the mean response, and the wider
# prediction interval for one new observation there, which adds the residual variance.
bands = model.get_prediction().summary_frame(alpha=1 - confidence)
intervals = pd.DataFrame({
    "no": np.arange(1, n + 1), "predicted": model.fittedvalues,
    "mean lower": bands["mean_ci_lower"], "mean upper": bands["mean_ci_upper"],
    "prediction lower": bands["obs_ci_lower"], "prediction upper": bands["obs_ci_upper"],
})
print()
print(intervals.to_string(index=False))

## Tests for heteroscedasticity

In [ ]:
# Regression assumes the residuals spread equally at every value of the explanatory
# variables. These tests ask whether the squared residuals rise or fall with them; a
# small P is a warning that the standard errors and P values above may be off (a
# large one does not prove the spread is equal).
# Breusch-Pagan in Koenker's form, het_breuschpagan()'s default, which does not assume
# normal residuals: n·R² of the squared residuals regressed on the variables.
# White's adds their squares and cross-products, k + k(k+1)/2 terms -- fewer when a
# product repeats a column, as a 0/1 variable's square does, so its df is the rank of
# that design less 1, the df het_white's P is computed on.
# Both auxiliary regressions keep a constant of their own, though the model has none.
exog = sm.add_constant(model.model.exog, has_constant="add")
bp = het_breuschpagan(model.resid, exog)
white = het_white(model.resid, exog)
upper = np.triu_indices(exog.shape[1])
white_df = np.linalg.matrix_rank(exog[:, upper[0]] * exog[:, upper[1]]) - 1
hetero = pd.DataFrame([[bp[0], k, bp[1]], [white[0], white_df, white[1]]],
                      index=["Breusch-Pagan", "White"], columns=["chi-square", "df", "P"])
print()
print(hetero)

## What the result says

This run keeps all three predictors and sets the constant to 0. That forces the model to predict zero sales for a shop with no foot traffic, no seats and no followers.

R squared jumps to 0.987, but do not read that as a better fit. Without a constant, R squared is measured from zero and not from the mean of sales, so it comes out much higher.

The coefficients still agree with the main finding. Foot traffic (3.07, P almost zero) and seats (0.37, P = 0.0004) are clearly linked with sales: a small P value means chance alone would rarely give such a link. Social followers have a coefficient of 0.11 with P = 0.84, and a confidence interval, the range of values the data fit, from -1.00 to 1.21, with zero in the middle. So even in this model, once you know foot traffic and seats, social followers add nothing these shops can detect.

## Your own data

Open the **Files** panel (the folder icon on the left) and upload a CSV file. Then change `"data.csv"` in the `read_csv` line to its name, change the column names in the code to yours, and choose **Run all** again.